# 1. Descarga de datos del Allen Institute (Visual Behavior)
Tres pasos, en este orden: **A** entorno (Python 3.10 + AllenSDK), **B** sesiones de entrenamiento (conducta), **C** velocidad de carrera, **D** pupila de los mismos ratones. Cada paso es reanudable y deja UNA sola celda corriendo. Al final de cada uno se descarga un zip (`data_train.zip`, `velocidad.zip`, `data_pupil.zip`).

In [ ]:
# A) Entorno (solo si no existe)
!ls /content/venv/bin/python || (pip install -q uv && uv venv --clear --python 3.10 /content/venv && uv pip install --python /content/venv/bin/python allensdk && uv pip install --python /content/venv/bin/python pynwb==2.5.0 hdmf==3.9.0)

## B) Sesiones de entrenamiento

In [ ]:
%%writefile fetch_allen_training.py
"""Download TRAINING (behavior-only) image-change-detection sessions of Allen Visual Behavior.

Why training sessions: they are behaviour-only NWB files (small, fast) and contain the acquisition
of the task (learning dynamics). Ophys sessions force the SDK to download the whole 2-photon file.

Usage:  python fetch_allen_training.py --out data_train --mice 12 --per_mouse 30
Output: data_train/sessions.csv  +  data_train/trials_<behavior_session_id>.csv   (resumable)
"""
import argparse, glob, os, sys
import pandas as pd


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data_train")
    ap.add_argument("--cache", default="allen_cache")
    ap.add_argument("--mice", type=int, default=8)
    ap.add_argument("--per_mouse", type=int, default=20)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    from allensdk.brain_observatory.behavior.behavior_project_cache import VisualBehaviorOphysProjectCache
    cache = VisualBehaviorOphysProjectCache.from_s3_cache(cache_dir=a.cache)
    bst = cache.get_behavior_session_table().reset_index()
    st = bst["session_type"].astype(str)
    ok = st.str.startswith("TRAINING") & st.str.contains("images") & bst["file_id"].notna()
    if "behavior_type" in bst.columns:
        ok &= ~bst["behavior_type"].astype(str).str.lower().str.contains("passive")
    t = bst[ok].copy()
    t["date"] = pd.to_datetime(t["date_of_acquisition"], utc=True, errors="coerce")
    t = t.sort_values(["mouse_id", "date"])
    cnt = t.groupby("mouse_id").size()
    mice = list(cnt[cnt >= 12].sort_values(ascending=False).index)[: a.mice]
    t = t[t["mouse_id"].isin(mice)].groupby("mouse_id", group_keys=False).head(a.per_mouse).copy()
    t["novel_images"] = 0
    t["order_in_mouse"] = t.groupby("mouse_id").cumcount()
    print("candidate training sessions:", int(ok.sum()), "| mice kept:", len(mice), "| sessions:", len(t))
    print(t["session_type"].value_counts().head(10).to_dict())
    if len(t) == 0:
        sys.exit("No sessions selected; paste the lines above.")
    keep = [c for c in ["behavior_session_id", "mouse_id", "session_type", "novel_images", "order_in_mouse",
                        "project_code", "date_of_acquisition", "cre_line", "file_id"] if c in t.columns]
    t[keep].to_csv(os.path.join(a.out, "sessions.csv"), index=False)
    cols = ["start_time", "stop_time", "go", "catch", "hit", "miss", "false_alarm", "correct_reject",
            "aborted", "auto_rewarded", "reward_volume", "response_latency", "change_time",
            "initial_image_name", "change_image_name"]
    n_ok = 0
    for i, bid in enumerate(t["behavior_session_id"]):
        path = os.path.join(a.out, "trials_%d.csv" % bid)
        if os.path.exists(path):
            n_ok += 1
            continue
        def purge(b):  # remove cached NWB (partial or not) to free disk / force a clean re-download
            for q in glob.glob(os.path.join(a.cache, "**", "*%d*.nwb" % b), recursive=True):
                try:
                    os.remove(q)
                except OSError:
                    pass
        for attempt in (1, 2):
            try:
                tr = cache.get_behavior_session(int(bid)).trials.reset_index()
                tr[[c for c in cols if c in tr.columns]].to_csv(path, index=False)
                n_ok += 1
                print("ok %d/%d  %d  trials=%d" % (n_ok, len(t), bid, len(tr)), flush=True)
                purge(bid)  # each file is ~190 MB: do not keep it
                break
            except Exception as e:
                purge(bid)
                if attempt == 2:
                    print("FAILED", bid, repr(e)[:200], file=sys.stderr, flush=True)


if __name__ == "__main__":
    main()


In [ ]:
!/content/venv/bin/python -u fetch_allen_training.py --out data_train --mice 30 --per_mouse 12

In [ ]:
!ls data_train | wc -l
!zip -q -r data_train.zip data_train
from google.colab import files
files.download('data_train.zip')

## C) Velocidad de carrera (mismas sesiones)
Sube `sessions_used.csv` (viene en el suplemento).

In [ ]:
import os, shutil
from google.colab import files
os.makedirs('data_train', exist_ok=True)
up = files.upload()
if not os.path.exists('data_train/sessions.csv'): shutil.copy(list(up)[0],'data_train/sessions.csv')

In [ ]:
%%writefile fetch_running.py
"""Add RUNNING SPEED (an arousal proxy that IS stored in behavior-only sessions) to the sessions already analyzed.
Pupil/eye tracking exists only in imaging sessions, so it is NOT available here.

For each selected session this writes data_train/run_<behavior_session_id>.csv with one row per row of
trials_<id>.csv (same order): run_pre = mean speed (cm/s) in [start_time, start_time+1.5 s]  (before any response),
                              run_trial = mean speed in [start_time, stop_time].
Usage (same environment as fetch_allen_training.py):
    python fetch_running.py --data data_train --per_mouse 4
Resumable, purges each NWB after use (~190 MB each).  UNTESTED against the live AllenSDK (the sandbox cannot reach it):
the speed table is read defensively (columns 'timestamps'/'speed' or index/values).
"""
import argparse, glob, os, sys
import numpy as np, pandas as pd


def speed_table(sess):
    rs = sess.running_speed
    if isinstance(rs, pd.DataFrame):
        cols = {c.lower(): c for c in rs.columns}
        ts = rs[cols["timestamps"]].to_numpy() if "timestamps" in cols else rs.index.to_numpy()
        sp = rs[cols["speed"]].to_numpy() if "speed" in cols else rs.iloc[:, 0].to_numpy()
    else:
        ts, sp = np.asarray(rs.index), np.asarray(rs.values)
    return np.asarray(ts, float), np.asarray(sp, float)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data_train"); ap.add_argument("--cache", default="allen_cache")
    ap.add_argument("--per_mouse", type=int, default=4)
    a = ap.parse_args()
    from allensdk.brain_observatory.behavior.behavior_project_cache import VisualBehaviorOphysProjectCache
    cache = VisualBehaviorOphysProjectCache.from_s3_cache(cache_dir=a.cache)
    s = pd.read_csv(os.path.join(a.data, "sessions.csv"))
    s = s.sort_values(["mouse_id", "order_in_mouse"]).groupby("mouse_id", group_keys=False).head(a.per_mouse)
    n_ok = 0
    for bid in s["behavior_session_id"]:
        out = os.path.join(a.data, "run_%d.csv" % bid)
        if os.path.exists(out):
            n_ok += 1; continue
        def purge():
            for q in glob.glob(os.path.join(a.cache, "**", "*%d*.nwb" % bid), recursive=True):
                try: os.remove(q)
                except OSError: pass
        for attempt in (1, 2):
            try:
                sess = cache.get_behavior_session(int(bid))
                tr = sess.trials.reset_index(); ts, sp = speed_table(sess)
                st = tr["start_time"].to_numpy(float); sp_end = tr["stop_time"].to_numpy(float)
                def mean_in(a0, a1):
                    i0, i1 = np.searchsorted(ts, a0), np.searchsorted(ts, a1)
                    return np.array([sp[x:y].mean() if y > x else np.nan for x, y in zip(i0, i1)])
                pd.DataFrame({"run_pre": mean_in(st, st + 1.5), "run_trial": mean_in(st, sp_end)}).to_csv(out, index=False)
                n_ok += 1; print("ok %d/%d %d" % (n_ok, len(s), bid), flush=True); purge(); break
            except Exception as e:
                purge()
                if attempt == 2: print("FAILED", bid, repr(e)[:200], file=sys.stderr, flush=True)

if __name__ == "__main__":
    main()


In [ ]:
!/content/venv/bin/python -u fetch_running.py --data data_train --per_mouse 4

In [ ]:
!zip -q velocidad.zip data_train/run_*.csv
from google.colab import files
files.download('velocidad.zip')

## D) Pupila de los mismos ratones (sesiones de imagen)
Usa `sessions_used.csv` ya subido (copiar a la carpeta de trabajo). Archivos grandes, máx. 40 sesiones.

In [ ]:
%%writefile fetch_pupil.py
"""Pupil area + running speed per trial from Allen Visual Behavior OPHYS sessions (the only sessions with eye tracking).

Selects ACTIVE ophys sessions (OPHYS_1/3/4/6), one experiment per behavior session, mice with >= --per_mouse active sessions.
Per session writes data_pupil/trials_<behavior_session_id>.csv (same columns as the training-data files) and
data_pupil/pupil_<behavior_session_id>.csv with, one row per trial row:
   pupil_pre : mean pupil area in [start_time, start_time+1.5 s] excluding likely blinks   (arousal readout)
   run_pre   : mean running speed (cm/s) in the same window
Each ophys NWB is LARGE (the SDK downloads the whole 2-photon file): the file is deleted after use.  Resumable.
UNTESTED against the live AllenSDK (the sandbox cannot reach it): eye-tracking columns are read defensively.
Usage: python fetch_pupil.py --out data_pupil --mice 8 --per_mouse 3
   Same animals as the behavior analysis: python fetch_pupil.py --out data_pupil --mice_csv sessions_used.csv --per_mouse 4 --min_sessions 2
"""
import argparse, glob, os, sys
import numpy as np, pandas as pd

COLS = ["start_time", "stop_time", "go", "catch", "hit", "miss", "false_alarm", "correct_reject", "aborted",
        "auto_rewarded", "reward_volume", "response_latency", "change_time", "initial_image_name", "change_image_name"]


def mean_in(ts, x, a0, a1):
    i0, i1 = np.searchsorted(ts, a0), np.searchsorted(ts, a1)
    return np.array([np.nanmean(x[p:q]) if q > p and np.isfinite(x[p:q]).any() else np.nan for p, q in zip(i0, i1)])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data_pupil"); ap.add_argument("--cache", default="allen_cache")
    ap.add_argument("--mice", type=int, default=8); ap.add_argument("--per_mouse", type=int, default=3)
    ap.add_argument("--mice_csv", default=None, help="sessions.csv of the behavior analysis: restrict to the SAME mice (column mouse_id)")
    ap.add_argument("--min_sessions", type=int, default=0); ap.add_argument("--max_sessions", type=int, default=999)
    a = ap.parse_args(); os.makedirs(a.out, exist_ok=True)
    from allensdk.brain_observatory.behavior.behavior_project_cache import VisualBehaviorOphysProjectCache
    cache = VisualBehaviorOphysProjectCache.from_s3_cache(cache_dir=a.cache)
    ex = cache.get_ophys_experiment_table().reset_index()
    print("columns:", list(ex.columns)[:25])
    st = ex["session_type"].astype(str)
    active = st.str.upper().str.startswith("OPHYS") & ~st.str.contains("passive", case=False) & ~st.str.contains("OPHYS_2|OPHYS_5")
    e = ex[active].drop_duplicates("behavior_session_id").copy()
    e["mouse_id"] = pd.to_numeric(e["mouse_id"], errors="coerce").astype("Int64").astype(int)  # ids may be strings in the table
    e["date"] = pd.to_datetime(e["date_of_acquisition"], utc=True, errors="coerce")
    e = e.sort_values(["mouse_id", "date"])
    cnt = e.groupby("mouse_id").size()
    if a.mice_csv:  # same animals as the behavior analysis
        want = set(int(x) for x in pd.read_csv(a.mice_csv)["mouse_id"]); cnt = cnt[cnt.index.isin(want)]
        print("mice of the behavior analysis with active ophys sessions:", len(cnt), "of", len(want), flush=True)
        mice = list(cnt[cnt >= max(a.min_sessions, 1)].sort_values(ascending=False).index)
    else:
        mice = list(cnt[cnt >= a.per_mouse].sort_values(ascending=False).index)[: a.mice]
    e = e[e["mouse_id"].isin(mice)].groupby("mouse_id", group_keys=False).head(a.per_mouse).copy()
    e = e.head(a.max_sessions).copy()
    e["novel_images"] = (~e["experience_level"].astype(str).str.lower().str.contains("familiar")).astype(int) if "experience_level" in e.columns else 0
    e["order_in_mouse"] = e.groupby("mouse_id").cumcount()
    print("active ophys sessions:", int(active.sum()), "| mice:", len(mice), "| sessions to fetch:", len(e), flush=True)
    keep = [c for c in ["behavior_session_id", "ophys_experiment_id", "mouse_id", "session_type", "novel_images", "order_in_mouse",
                        "date_of_acquisition", "cre_line"] if c in e.columns]
    sp = os.path.join(a.out, "sessions.csv")
    out_t = e[keep]
    if os.path.exists(sp):  # keep sessions fetched in earlier runs
        out_t = pd.concat([pd.read_csv(sp), out_t]).drop_duplicates("behavior_session_id")
    out_t.to_csv(sp, index=False)
    n_ok = 0
    for _, r in e.iterrows():
        bid, eid = int(r["behavior_session_id"]), int(r["ophys_experiment_id"])
        if os.path.exists(os.path.join(a.out, "pupil_%d.csv" % bid)):
            n_ok += 1; continue
        def purge():
            for q in glob.glob(os.path.join(a.cache, "**", "*%d*.nwb" % eid), recursive=True) + glob.glob(os.path.join(a.cache, "**", "*%d*.nwb" % bid), recursive=True):
                try: os.remove(q)
                except OSError: pass
        for attempt in (1, 2):
            try:
                exp = cache.get_behavior_ophys_experiment(eid)
                tr = exp.trials.reset_index(); et = exp.eye_tracking; rs = exp.running_speed
                ets = et["timestamps"].to_numpy(float) if "timestamps" in et.columns else et.index.to_numpy(float)
                area = et["pupil_area"].to_numpy(float).copy()
                if "likely_blink" in et.columns: area[et["likely_blink"].to_numpy(bool)] = np.nan
                rts = rs["timestamps"].to_numpy(float) if "timestamps" in rs.columns else rs.index.to_numpy(float)
                rsp = rs["speed"].to_numpy(float)
                s0 = tr["start_time"].to_numpy(float)
                pd.DataFrame({"pupil_pre": mean_in(ets, area, s0, s0 + 1.5), "run_pre": mean_in(rts, rsp, s0, s0 + 1.5)}).to_csv(
                    os.path.join(a.out, "pupil_%d.csv" % bid), index=False)
                tr[[c for c in COLS if c in tr.columns]].to_csv(os.path.join(a.out, "trials_%d.csv" % bid), index=False)
                n_ok += 1; print("ok %d/%d %d trials=%d" % (n_ok, len(e), bid, len(tr)), flush=True); purge(); break
            except Exception as ex_:
                purge()
                if attempt == 2: print("FAILED", bid, repr(ex_)[:300], file=sys.stderr, flush=True)


if __name__ == "__main__":
    main()


In [ ]:
!/content/venv/bin/python -u fetch_pupil.py --out data_pupil --mice_csv sessions_used.csv --per_mouse 4 --min_sessions 2 --max_sessions 40

In [ ]:
!zip -q -r data_pupil.zip data_pupil
from google.colab import files
files.download('data_pupil.zip')